In [4]:
import numpy as np 
import pandas as pd

cols = [
	"age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
	"thalach", "exang", "oldpeak", "slope", "ca", "thal", "target"
]
df = pd.read_csv("processed.cleveland.data", names=cols, na_values="?")
print(df.shape)
print(".....................................")
print(df.isna().sum())
print(".....................................")
print(df["target"].value_counts())



(303, 14)
.....................................
age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          4
thal        2
target      0
dtype: int64
.....................................
target
0    164
1     55
2     36
3     35
4     13
Name: count, dtype: int64


In [5]:
df["target"] = (df["target"] > 0).astype(int)
print(df["target"].value_counts())

target
0    164
1    139
Name: count, dtype: int64


In [6]:
random_number_generator = np.random.default_rng(42)
index = random_number_generator.permutation(len(df))
split = int(0.2 * len(df))

test_df = df.iloc[index[:split]].reset_index(drop=True)
train_df = df.iloc[index[split:]].reset_index(drop=True)

print(train_df.shape, test_df.shape)

(243, 14) (60, 14)


In [7]:
for col in (df[["ca", "thal"]]):
    fill_value = train_df[col].median()
    train_df[col] = train_df[col].fillna(fill_value)
    test_df[col] = test_df[col].fillna(fill_value)
    print(col, fill_value)

print(train_df.isna().sum().sum(), test_df.isna().sum().sum())

ca 0.0
thal 3.0
0 0


In [8]:
cat_cols = ["cp" , "restecg", "slope", "thal"]

train_df = pd.get_dummies(train_df, columns=cat_cols, dtype=float)
print(train_df.shape)

test_df = pd.get_dummies(test_df, columns=cat_cols, dtype=float)
test_df = test_df.reindex(columns = train_df.columns, fill_value = 0.0)

print(test_df.shape)


(243, 23)
(60, 23)


In [9]:
print(list(train_df.columns) == list(test_df.columns))

True


In [10]:
num_cols = ["age", "trestbps", "chol", "thalach", "oldpeak", "ca"]
print(train_df[num_cols].describe())

print("..........................................")
train_mean = train_df[num_cols].mean()
train_std = train_df[num_cols].std()
print(train_mean)
print(train_std)

train_df[num_cols] = (train_df[num_cols] - train_mean) / train_std
test_df[num_cols] = (test_df[num_cols] - train_mean) / train_std

              age    trestbps        chol     thalach     oldpeak          ca
count  243.000000  243.000000  243.000000  243.000000  243.000000  243.000000
mean    54.913580  131.366255  246.872428  149.493827    1.020165    0.658436
std      8.907046   16.926726   47.122096   23.220533    1.158139    0.941798
min     29.000000   94.000000  141.000000   71.000000    0.000000    0.000000
25%     48.000000  120.000000  212.500000  132.500000    0.000000    0.000000
50%     56.000000  130.000000  243.000000  153.000000    0.800000    0.000000
75%     61.000000  140.000000  274.000000  166.500000    1.600000    1.000000
max     77.000000  192.000000  417.000000  202.000000    6.200000    3.000000
..........................................
age          54.913580
trestbps    131.366255
chol        246.872428
thalach     149.493827
oldpeak       1.020165
ca            0.658436
dtype: float64
age          8.907046
trestbps    16.926726
chol        47.122096
thalach     23.220533
oldpeak      1

In [11]:
print(train_df[num_cols].mean().round(3))
print(train_df[num_cols].std().round(3))
print(test_df[num_cols].mean().round(3))


age        -0.0
trestbps    0.0
chol       -0.0
thalach     0.0
oldpeak     0.0
ca         -0.0
dtype: float64
age         1.0
trestbps    1.0
chol        1.0
thalach     1.0
oldpeak     1.0
ca          1.0
dtype: float64
age        -0.269
trestbps    0.097
chol       -0.019
thalach     0.025
oldpeak     0.085
ca          0.026
dtype: float64


In [12]:
x_train = train_df.drop(columns=["target"]).to_numpy()
y_train = train_df["target"].to_numpy()

x_test = test_df.drop(columns = "target").to_numpy()
y_test = test_df["target"].to_numpy()

print(x_train.shape, y_train.shape)
print(x_test.shape, y_test.shape)
print("..........................................")
print("train mean: ", y_train.mean())
print("test mean: ", y_test.mean())

(243, 22) (243,)
(60, 22) (60,)
..........................................
train mean:  0.4732510288065844
test mean:  0.4


In [13]:
def evaluate(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))

    total = tp + tn + fp + fn
    accuracy = (tp + tn) / total if total > 0 else 0

    precision = tp / (tp + fp) if tp + fp > 0 else 0
    recall = tp / (tp + fn) if tp + fn > 0 else 0

    if precision + recall > 0:
        f1 = 2 * precision * recall / (precision + recall)
    else:
        f1 = 0

    results = {
        "TP": int(tp),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "Accuracy": round(accuracy, 4),
        "Precision": round(precision, 4),
        "Recall": round(recall, 4),
        "F1": round(f1, 4)
    }

    return results

In [15]:
t = np.array([1, 1, 1, 0, 0, 0, 0, 1])
p = np.array([1, 1, 0, 0, 0, 1, 1, 1])
print(evaluate(t, p))

print(evaluate(y_test, np.zeros(len(y_test), dtype=int)))

{'TP': 3, 'TN': 2, 'FP': 2, 'FN': 1, 'Accuracy': np.float64(0.625), 'Precision': np.float64(0.6), 'Recall': np.float64(0.75), 'F1': np.float64(0.6667)}
{'TP': 0, 'TN': 36, 'FP': 0, 'FN': 24, 'Accuracy': np.float64(0.6), 'Precision': 0, 'Recall': np.float64(0.0), 'F1': 0}


In [21]:
class GaussianNaiveBayes:
    def __init__(self, var_smoothing=1e-9):
        self.var_smoothing = var_smoothing
        
    def _posterior(self, X, c):
        mean, var = self.means[c], self.vars[c]
        scores = (1 / np.sqrt(2 * np.pi * var)) * np.exp(-((X - mean) ** 2) / (2 * var))
        return self.priors[c] * np.prod(scores, axis=1)

    def predict(self, X):
        all_scores = np.column_stack([self._posterior(X, c) for c in self.classes])
        return self.classes[np.argmax(all_scores, axis=1)]    

    def fit(self, X, y):
        self.classes = np.unique(y)

        self.priors = {}
        self.means = {}
        self.vars = {}

        epsilon = self.var_smoothing * np.var(X, axis=0).max()

        for c in self.classes:
            X_c = X[y == c]

            self.priors[c] = len(X_c) / len(X)
            self.means[c] = X_c.mean(axis=0)
            self.vars[c] = X_c.var(axis=0) + epsilon

        return self

In [22]:
nb = GaussianNaiveBayes()
nb.fit(x_train, y_train)
y_pred = nb.predict(x_test)
print(evaluate(y_test, y_pred))
print(evaluate(y_train, nb.predict(x_train)))

{'TP': 20, 'TN': 26, 'FP': 10, 'FN': 4, 'Accuracy': np.float64(0.7667), 'Precision': np.float64(0.6667), 'Recall': np.float64(0.8333), 'F1': np.float64(0.7407)}
{'TP': 95, 'TN': 113, 'FP': 15, 'FN': 20, 'Accuracy': np.float64(0.856), 'Precision': np.float64(0.8636), 'Recall': np.float64(0.8261), 'F1': np.float64(0.8444)}


In [ ]:
class MyLogisticRegression:
    def __init__(self, learning_rate=0.1, epochs=1000):
        self.learning_rate = learning_rate   
        self.epochs = epochs               
        self.weights = None
        self.bias = 0

    def sigmoid(self, z):
        return 1 / (1 + np.exp(-z))

    def fit(self, X, y):
        n_samples = X.shape[0]
        n_features = X.shape[1]

        
        self.weights = np.zeros(n_features)
        self.bias = 0

        for i in range(self.epochs):
            
            z = np.dot(X, self.weights) + self.bias

            predictions = self.sigmoid(z)

            
            errors = predictions - y

            
            dw = np.dot(X.T, errors) / n_samples
            db = np.sum(errors) / n_samples

        
            self.weights = self.weights - self.learning_rate * dw
            self.bias = self.bias - self.learning_rate * db

    def predict_proba(self, X):
        z = np.dot(X, self.weights) + self.bias
        return self.sigmoid(z)

    def predict(self, X):
        probabilities = self.predict_proba(X)
        return (probabilities >= 0.5).astype(int)  

In [30]:
lr_model = MyLogisticRegression()
lr_model.fit(x_train, y_train)

print(evaluate(y_test, lr_model.predict(x_test)))
print(evaluate(y_train, lr_model.predict(x_train)))

{'TP': 20, 'TN': 28, 'FP': 8, 'FN': 4, 'Accuracy': np.float64(0.8), 'Precision': np.float64(0.7143), 'Recall': np.float64(0.8333), 'F1': np.float64(0.7692)}
{'TP': 95, 'TN': 114, 'FP': 14, 'FN': 20, 'Accuracy': np.float64(0.8601), 'Precision': np.float64(0.8716), 'Recall': np.float64(0.8261), 'F1': np.float64(0.8482)}
